# WealthGen — Load reference dataset into a Fabric Lakehouse

Loads the 11 synthetic advisory tables (`clients`, `mandates`, `holdings`, `attribution`,
`portfolio_performance`, market context, etc.) into **managed Delta tables** in the attached
Lakehouse. The Lakehouse **SQL analytics endpoint** then exposes them as `dbo.<table>`, which
the backend reads via `DATA_SOURCE_MODE=fabric` (`app.services.fabric_data`, SELECT-only).

## Prerequisites
1. **Attach a Lakehouse** to this notebook (Explorer ▸ *Add Lakehouse* ▸ pick/create one).
2. **Generate the CSVs** locally (once):
   ```powershell
   cd backend
   python -m scripts.synthetic.generate
   ```
   This writes `backend/data/synthetic/fabric_iq/*.csv` (11 files).
3. **Upload the CSVs** into the Lakehouse **Files** area under a `wealthgen/` folder
   (Lakehouse ▸ Files ▸ *Upload* ▸ upload all 11 `*.csv`). The default expected path is
   `Files/wealthgen/<table>.csv`.

After running, point the backend at this Lakehouse's SQL analytics endpoint:
```
DATA_SOURCE_MODE=fabric
FABRIC_SQL_ENDPOINT=<lakehouse-sql-endpoint>.datawarehouse.fabric.microsoft.com
FABRIC_DATABASE=<lakehouse-name>
FABRIC_SQL_SCHEMA=dbo
```

## 1. Configuration

Table list and the typed-column sets mirror `backend/scripts/fabric/schema.sql` so the Delta
column types (double / boolean) read back correctly through the SQL endpoint
(`fabric_data._normalize`: `bool → "True"/"False"`, numeric → str).

In [ ]:
# Folder in the Lakehouse Files area where the 11 CSVs were uploaded.
FILES_DIR = "Files/wealthgen"

# The 11 advisory reference tables (load order is cosmetic; no FKs).
TABLES = [
    "advisors",
    "clients",
    "mandates",
    "benchmarks",
    "holdings",
    "portfolio_performance",
    "attribution",
    "positioning_changes",
    "market_index_returns",
    "fx_moves",
    "vix_events",
]

# Columns stored as FLOAT/DECIMAL in schema.sql -> Delta double.
FLOAT_COLS = {
    "weight", "aum_musd", "market_value_usd", "period_return_pct",
    "total_return_net_pct", "benchmark_return_pct", "active_return_bps",
    "tracking_error_pct", "information_ratio", "ex_ante_vol_pct", "sharpe",
    "max_drawdown_pct", "portfolio_weight", "benchmark_weight",
    "portfolio_return", "benchmark_return", "allocation_bps", "selection_bps",
    "interaction_bps", "change_pct", "vix_close",
}

# Columns stored as BIT in schema.sql -> Delta boolean.
BOOL_COLS = {"esg_preference", "event_trigger"}

print(f"Will load {len(TABLES)} tables from {FILES_DIR}")

## 2. Load each CSV into a Delta table

Reads each CSV as all-string (empty string → null), casts the known numeric/boolean columns,
and writes a **managed** Delta table with `overwrite` (idempotent — safe to re-run).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType


def load_table(table: str) -> int:
    path = f"{FILES_DIR}/{table}.csv"
    df = (
        spark.read
        .option("header", True)
        .option("nullValue", "")
        .option("mode", "PERMISSIVE")
        .csv(path)  # all columns read as string
    )
    for c in df.columns:
        if c in FLOAT_COLS:
            df = df.withColumn(c, F.col(c).cast(DoubleType()))
        elif c in BOOL_COLS:
            df = df.withColumn(
                c,
                F.when(F.lower(F.col(c)).isin("true", "1"), F.lit(True))
                 .when(F.lower(F.col(c)).isin("false", "0"), F.lit(False))
                 .otherwise(F.lit(None)),
            )
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    return df.count()


results = {}
for t in TABLES:
    try:
        results[t] = load_table(t)
        print(f"{t:24s} {results[t]:5d} rows  [OK]")
    except Exception as exc:  # keep going; report at the end
        results[t] = f"FAILED: {exc}"
        print(f"{t:24s}  [FAILED]  {exc}")

print("\nDone.")

## 3. Verify

Confirm the tables exist and preview a couple. These are now queryable through the Lakehouse
**SQL analytics endpoint** as `dbo.<table>`.

In [ ]:
print("Row counts:")
for t in TABLES:
    print(f"  {t:24s} {spark.table(t).count():5d}")

display(spark.table("holdings").limit(5))
display(spark.table("portfolio_performance").limit(5))

## Next steps

1. **Wire the backend** — copy the Lakehouse SQL analytics endpoint host + name into `backend/.env`:
   ```
   DATA_SOURCE_MODE=fabric
   FABRIC_SQL_ENDPOINT=<sql-endpoint>.datawarehouse.fabric.microsoft.com
   FABRIC_DATABASE=<lakehouse-name>
   FABRIC_SQL_SCHEMA=dbo
   ```
   Then verify locally: `python -c "from app.services import fabric_data; print(fabric_data.query('SELECT COUNT(*) c FROM dbo.holdings'))"`
2. **Build a Fabric Data Agent** over this Lakehouse (for `GROUNDING_MODE=foundry_iq`) and set
   `FABRIC_WORKSPACE_ID` + `FABRIC_DATA_AGENT_ID` in `.env`.
3. Grant the app identity (service principal / your user) at least **Read** on the Lakehouse's
   SQL analytics endpoint so the AAD-token read path can connect.